In [4]:
import pandas as pd

from read_data import ImportTrainingData

X_events, y = ImportTrainingData()

In [ ]:
from torch.nn.utils.rnn import pad_sequence


def collate_sequence_batch(batch):
    """LLM assisted (UiOGPT)"""
    """
    Padder en batch med variable sekvenser.

    Returnerer
    ----------
    x_padded : torch.Tensor
        Shape: (batch_size, maks_sekvenslengde_i_batch, antall_features)

    lengths : torch.Tensor
        Shape: (batch_size,)
        Faktiske lengder før padding.

    y : torch.Tensor
        Shape: (batch_size,)
    """
    sequences, targets = zip(*batch)

    lengths = torch.tensor(
        [len(sequence) for sequence in sequences],
        dtype=torch.long,
    )

    # batch_first=True gir:
    # (batch_size, maks_lengde, antall_features)
    x_padded = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=0.0,
    )

    y = torch.stack(targets)

    return x_padded, lengths, y


In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset


class SequenceDataFrameDataset(Dataset):
    def __init__(
        self,
        dataframes,
        targets=None,
        object_type_column="c_object_type",
        categories=None,
        dtype=torch.float32,
    ):
        """LLM assisted (UiOGPT)"""
        """
        Parametere
        ----------
        dataframes : list[pd.DataFrame]
            Ett DataFrame per sekvens.
            Hver rad er ett tidsteg.

        targets : array-like eller None
            Én target per DataFrame/sekvens.
            For eksempel en liste med risikoer.

        object_type_column : str
            Kolonnen som skal one-hot-encodes.

        categories : list[str] eller None
            Kategorier for c_object_type.
            Dersom None: finn kategorier fra dataframes.
            Ved validering/test: send inn kategoriene fra treningssettet.

        dtype : torch.dtype
            Datatype for input og target.
        """
        if len(dataframes) == 0:
            raise ValueError("dataframes kan ikke være tom.")

        if targets is not None and len(dataframes) != len(targets):
            raise ValueError(
                "dataframes og targets må ha samme antall elementer."
            )

        self.dataframes = dataframes
        self.targets = targets
        self.object_type_column = object_type_column
        self.dtype = dtype

        # Kontroller kolonner og hent numeriske feature-kolonner
        first_df = dataframes[0]

        if object_type_column not in first_df.columns:
            raise KeyError(
                f"Kolonnen '{object_type_column}' finnes ikke."
            )

        self.original_columns = list(first_df.columns)

        self.numeric_columns = [
            col for col in self.original_columns
            if col != object_type_column
        ]

        for i, df in enumerate(dataframes):
            if len(df) == 0:
                raise ValueError(f"DataFrame med indeks {i} er tomt.")

            if list(df.columns) != self.original_columns:
                raise ValueError(
                    f"DataFrame {i} har andre eller feil sorterte kolonner."
                )

        # Finn kategorier fra første rad i hvert DataFrame.
        # Kun nødvendig fordi c_object_type er konstant i en sekvens.
        if categories is None:
            categories = sorted({
                str(df[object_type_column].iloc[0])
                for df in dataframes
            })

        self.categories = list(categories)

        self.category_to_index = {
            category: index
            for index, category in enumerate(self.categories)
        }

        self.feature_names = (
            self.numeric_columns
            + [f"{object_type_column}__{category}"
               for category in self.categories]
        )

    def __len__(self):
        return len(self.dataframes)

    def __getitem__(self, index):
        df = self.dataframes[index]

        # Numeriske kolonner: shape = (sekvenslengde, antall_numeriske_features)
        numeric_values = df[self.numeric_columns].to_numpy(
            dtype=np.float32
        )

        sequence_length = len(df)

        # Leser bare første objekt-type, ettersom den er lik for hele sekvensen
        object_type = str(df[self.object_type_column].iloc[0])

        # Dersom en ukjent kategori dukker opp i validering/test,
        # får den en ren nullvektor.
        one_hot = np.zeros(
            (sequence_length, len(self.categories)),
            dtype=np.float32,
        )

        if object_type in self.category_to_index:
            category_index = self.category_to_index[object_type]
            one_hot[:, category_index] = 1.0
        else:
            print(
                f"Advarsel: Ukjent c_object_type '{object_type}' "
                f"i DataFrame {index}."
            )

        # Kombinerer numeriske og one-hot-features
        sequence = np.concatenate(
            [numeric_values, one_hot],
            axis=1,
        )

        # PyTorch-tensor med shape:
        # (sekvenslengde, antall_features)
        x = torch.tensor(sequence, dtype=self.dtype)

        if self.targets is None:
            return x

        # Én target per sekvens, f.eks. risiko
        y = torch.tensor(
            self.targets[index],
            dtype=self.dtype,
        )

        return x, y


In [ ]:

import numpy as np #<- Skrevet av UiOGPT

def dataframes_til_sekvenser(
    dataframes,
    pad_value=0.0,
    dtype=np.float32,
    object_type_column="c_object_type",
    return_categories=False,
):
    """LLM assisted (UiOGPT)"""
    """
    Konverterer en liste med DataFrames til et paddet 3D-array for RNN.

    c_object_type one-hot-encodes automatisk basert på alle unike verdier
    som finnes i første rad av hvert DataFrame.

    Parametere
    ----------
    dataframes : list[pd.DataFrame]
        Hvert DataFrame representerer én sekvens.
        Rader = tidspunkter.
        Kolonner = input-features.

    pad_value : float
        Verdi brukt for padding.

    dtype : numpy dtype
        Datatype på output-arrayet.

    object_type_column : str
        Navnet på kolonnen som skal one-hot-encodes.

    return_categories : bool
        Hvis True returneres også listen med kategorier.

    Returnerer
    ----------
    X : np.ndarray
        Shape: (batch_size, maks_sekvenslengde, antall_features)

    lengths : np.ndarray
        Faktisk lengde på hver sekvens.

    categories : list[str], optional
        Returneres bare dersom return_categories=True.
    """
    if not dataframes:
        raise ValueError("Listen med DataFrames kan ikke være tom.")

    # Kontroller at alle DataFrames har nødvendig kolonne og minst én rad
    for i, df in enumerate(dataframes):
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"Element {i} er ikke et pandas DataFrame.")

        if len(df) == 0:
            raise ValueError(f"DataFrame {i} er tomt.")

        if object_type_column not in df.columns:
            raise KeyError(
                f"DataFrame {i} mangler kolonnen '{object_type_column}'."
            )

    # Leser kun første rad fra hvert DataFrame, siden verdien er den samme
    # gjennom hele sekvensen.
    categories = sorted({
        str(df[object_type_column].iloc[0])
        for df in dataframes
    })

    category_to_index = {
        category: index
        for index, category in enumerate(categories)
    }

    print(f"Fant {len(categories)} kategorier i '{object_type_column}':")
    print(categories)

    # Alle vanlige features, uten strenge-kolonnen som erstattes av one-hot.
    vanlige_kolonner = [
        col for col in dataframes[0].columns
        if col != object_type_column
    ]

    # Kontroller at alle DataFrames har samme kolonner
    forventede_kolonner = list(dataframes[0].columns)

    for i, df in enumerate(dataframes):
        if list(df.columns) != forventede_kolonner:
            raise ValueError(
                f"DataFrame {i} har andre eller feil sorterte kolonner."
            )

    lengths = np.array([len(df) for df in dataframes], dtype=np.int64)

    batch_size = len(dataframes)
    maks_sekvenslengde = lengths.max()

    antall_vanlige_features = len(vanlige_kolonner)
    antall_one_hot_features = len(categories)
    antall_features = antall_vanlige_features + antall_one_hot_features

    X = np.full(
        shape=(batch_size, maks_sekvenslengde, antall_features),
        fill_value=pad_value,
        dtype=dtype,
    )

    for i, df in enumerate(dataframes):
        sekvenslengde = len(df)

        # Konverterer alle de vanlige, numeriske kolonnene
        numeriske_features = df[vanlige_kolonner].to_numpy(dtype=dtype)

        # Henter kategori fra første rad
        object_type = str(df[object_type_column].iloc[0])
        object_type_index = category_to_index[object_type]

        # Lager one-hot-matrise for hele sekvensen.
        # Samme kategori gjentas for alle rader i dette DataFrame-et.
        one_hot = np.zeros(
            (sekvenslengde, antall_one_hot_features),
            dtype=dtype,
        )
        one_hot[:, object_type_index] = 1.0

        # Kombinerer numeriske features og one-hot-features
        sekvens = np.concatenate(
            [numeriske_features, one_hot],
            axis=1,
        )

        X[i, :sekvenslengde, :] = sekvens

    if return_categories:
        return X, lengths, categories

    return X, lengths


In [6]:
X_train, lengths = dataframes_til_sekvenser(X_events)

Fant 5 kategorier i 'c_object_type':
['DEBRIS', 'PAYLOAD', 'ROCKET BODY', 'TBA', 'UNKNOWN']


c:\Users\Jon\anaconda3\envs\jaxenv\Lib\site-packages\pandas\core\internals\managers.py:1944: RuntimeWarning: overflow encountered in cast
  result[rl.indexer] = arr


In [ ]:
#Skrevet av ChatGPT:
import torch
import torch.nn as nn

class StackedRNN(nn.Module):
    """LLM assisted"""

    def __init__(self):
        """LLM assisted"""
        super().__init__()

        # RNN-lag 1: 3 input-features -> 16 hidden units
        self.rnn1 = nn.RNN(
            input_size=3,
            hidden_size=16,
            batch_first=True
        )

        # RNN-lag 2: 16 input-features -> 10 hidden units
        self.rnn2 = nn.RNN(
            input_size=16,
            hidden_size=10,
            batch_first=True
        )

        # RNN-lag 3: 10 input-features -> 5 hidden units
        self.rnn3 = nn.RNN(
            input_size=10,
            hidden_size=5,
            batch_first=True
        )

        # Output-lag: 5 -> 2 klasser
        self.fc = nn.Linear(5, 1)

    def forward(self, x):
        """LLM assisted"""

        # Første RNN-lag
        out1, hidden1 = self.rnn1(x)

        # Andre RNN-lag får output fra det første
        out2, hidden2 = self.rnn2(out1)

        # Tredje RNN-lag får output fra det andre
        out3, hidden3 = self.rnn3(out2)

        # Hent siste hidden state fra tredje RNN-lag
        final_hidden = hidden3[-1]

        # Lag prediksjon
        prediction = self.fc(final_hidden)

        return prediction


In [10]:
class F2VektetMSELoss(nn.Module):
    """LLM assisted (UiOGPT)"""
    def __init__(self, threshold=1e-6, beta=2.0, eps=1e-8):
        """LLM assisted (UiOGPT)"""
        super().__init__()
        self.threshold = threshold
        self.beta = beta
        self.eps = eps

    def beregn_f2(self, y_true, y_pred):
        """LLM assisted (UiOGPT)"""
        """
        Beregner F2-score basert på hele inputen.

        Positiv klasse:
            risiko >= threshold
        """
        y_true = y_true.reshape(-1)
        y_pred = y_pred.reshape(-1)

        ekte_positiv = y_true >= self.threshold
        predikert_positiv = y_pred >= self.threshold

        tp = (ekte_positiv & predikert_positiv).sum().float()
        fp = (~ekte_positiv & predikert_positiv).sum().float()
        fn = (ekte_positiv & ~predikert_positiv).sum().float()

        precision = tp / (tp + fp + self.eps)
        recall = tp / (tp + fn + self.eps)

        beta_kvadrat = self.beta ** 2

        f2 = (
            (1 + beta_kvadrat) * precision * recall
            / (beta_kvadrat * precision + recall + self.eps)
        )

        return f2

    def forward(self, y_pred, y_true, f2_score=None):
        """LLM assisted (UiOGPT)"""
        """
        Parameters
        ----------
        y_pred : torch.Tensor
            Predikert risiko, r_hat.

        y_true : torch.Tensor
            Fasiten / faktisk risiko, r.

        f2_score : float eller torch.Tensor, optional
            F2 beregnet over hele datasettet. Hvis ikke gitt,
            beregnes F2 fra batchen. For trening bør du helst
            sende inn en F2-score fra hele treningssettet.
        """
        y_pred = y_pred.reshape(-1)
        y_true = y_true.reshape(-1)

        # MSE skal bare gjelde den høye risikoklassen
        high_risk_mask = y_true >= self.threshold

        # Dersom batchen ikke har noen positive eksempler
        if not high_risk_mask.any():
            return y_pred.sum() * 0.0

        mse = torch.mean(
            (y_true[high_risk_mask] - y_pred[high_risk_mask]) ** 2
        )

        if f2_score is None:
            # detach fordi F2 bruker hard terskel og ikke er deriverbar
            with torch.no_grad():
                f2_score = self.beregn_f2(y_true, y_pred)

        if not torch.is_tensor(f2_score):
            f2_score = torch.tensor(
                f2_score,
                dtype=y_pred.dtype,
                device=y_pred.device,
            )
        else:
            f2_score = f2_score.to(
                dtype=y_pred.dtype,
                device=y_pred.device,
            )

        # Unngår divisjon på null
        f2_score = torch.clamp(f2_score, min=self.eps)

        loss = mse / f2_score

        return loss


In [ ]:
from torch.utils.data import Dataset, DataLoader

model = StackedRNN()

threshold = 10e-6
beta = 2

criterion = F2VektetMSELoss(beta = beta, theshold = threshold)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 20

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch_X, batch_y in train_loader:

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        predictions = model(batch_X)

        # Calculate loss
        loss = criterion(predictions, batch_y)

        # Backpropagation through time
        loss.backward()

        # Update weights
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{epochs}, "
        f"Loss: {average_loss:.4f}"
    )